In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
import sbdrank
import pandas as pd

from pathlib import Path


NOTEBOOK_DIR = Path().cwd()
DATA_DIR = NOTEBOOK_DIR.parent / "data" / "raw"
CSV_PATH = list(DATA_DIR.rglob("*.csv"))[0] # Only gets first .csv it finds

print(sbdrank.__file__)
print(CSV_PATH)

# Dataset Sample Exploration
First 100k rows of the dataset.
Doesn't appear specifically "ordered" by meet, but is grouped.

In [59]:
nrows = 100000
category_cols = ["Tested", "Sex", "Sanctioned", "Equipment", "BirthYearClass", 
                 "Event", "ParentFederation", "MeetCountry", "Federation"]

sample = pd.read_csv(
    CSV_PATH, 
    dtype={col: "category" for col in category_cols},
    parse_dates=["Date"],
    nrows=nrows)

for i, column in enumerate(sample.columns):
    print(f"COL {i} ({column.upper()})\t" + 
            f"Increasing: {sample[column].is_monotonic_increasing} " +
             f"Decreasing: {sample[column].is_monotonic_decreasing}")

COL 0 (NAME)	Increasing: False Decreasing: False
COL 1 (SEX)	Increasing: False Decreasing: False
COL 2 (EVENT)	Increasing: False Decreasing: False
COL 3 (EQUIPMENT)	Increasing: False Decreasing: False
COL 4 (AGE)	Increasing: False Decreasing: False
COL 5 (AGECLASS)	Increasing: False Decreasing: False
COL 6 (BIRTHYEARCLASS)	Increasing: False Decreasing: False
COL 7 (DIVISION)	Increasing: False Decreasing: False
COL 8 (BODYWEIGHTKG)	Increasing: False Decreasing: False
COL 9 (WEIGHTCLASSKG)	Increasing: False Decreasing: False
COL 10 (SQUAT1KG)	Increasing: False Decreasing: False
COL 11 (SQUAT2KG)	Increasing: False Decreasing: False
COL 12 (SQUAT3KG)	Increasing: False Decreasing: False
COL 13 (SQUAT4KG)	Increasing: False Decreasing: False
COL 14 (BEST3SQUATKG)	Increasing: False Decreasing: False
COL 15 (BENCH1KG)	Increasing: False Decreasing: False
COL 16 (BENCH2KG)	Increasing: False Decreasing: False
COL 17 (BENCH3KG)	Increasing: False Decreasing: False
COL 18 (BENCH4KG)	Increasing: False

/tmp/ipykernel_7251/1232975066.py:5: DtypeWarning: Columns (33: State, 38: MeetState) have mixed types. Specify dtype option on import or set low_memory=False.
  sample = pd.read_csv(


In [37]:
sample.nunique().sort_values()

Tested                  1
Sex                     2
Sanctioned              2
Equipment               5
BirthYearClass          7
Event                   7
ParentFederation       12
AgeClass               18
MeetCountry            48
Federation             56
State                  68
MeetState              76
WeightClassKg          82
Country               116
Place                 145
Age                   161
Squat4Kg              194
Deadlift4Kg           306
Bench4Kg              331
Division              492
MeetTown              504
Deadlift1Kg           514
Best3BenchKg          564
Best3DeadliftKg       584
Best3SquatKg          597
Squat1Kg              623
Bench1Kg              636
Deadlift2Kg           785
Squat2Kg              819
Bench2Kg              855
Date                  900
Bench3Kg              969
Squat3Kg              974
Deadlift3Kg           976
MeetName             1131
TotalKg              1678
BodyweightKg         7279
Goodlift             8512
Glossbrenner

In [41]:
for col in ["State", "MeetState"]:
    print(col)
    print(sample[col].map(type).value_counts().to_string())
    print()
    
    types = sample[col].map(type)
    odd = sample.loc[(types != str) & sample[col].notna(), col]
    print(col, "->", odd.unique()[:20])

State
State
<class 'float'>    75176
<class 'str'>      24824

State -> <StringArray>
[]
Length: 0, dtype: str
MeetState
MeetState
<class 'float'>    58243
<class 'str'>      41757

MeetState -> <StringArray>
[]
Length: 0, dtype: str


In [32]:
sample.isna().mean().sort_values(ascending=False)

Squat4Kg            0.99635
Bench4Kg            0.99068
Deadlift4Kg         0.98997
State               0.75176
MeetState           0.58243
Squat3Kg            0.40601
Tested              0.39258
Squat2Kg            0.39143
Squat1Kg            0.38352
Best3SquatKg        0.35692
Deadlift3Kg         0.32832
Deadlift2Kg         0.30472
Deadlift1Kg         0.29201
Best3DeadliftKg     0.25537
Bench3Kg            0.19541
ParentFederation    0.19113
Bench2Kg            0.17008
Bench1Kg            0.16067
Goodlift            0.15720
Age                 0.13697
BirthYearClass      0.13391
Best3BenchKg        0.12270
AgeClass            0.11951
MeetTown            0.09899
Wilks               0.04340
Glossbrenner        0.04340
Dots                0.04340
TotalKg             0.04195
Country             0.02149
WeightClassKg       0.01291
BodyweightKg        0.00335
Division            0.00026
Name                0.00000
Sex                 0.00000
Equipment           0.00000
Event               

# Full Data Load
Snapshot from 2026-11-21. Total rows = 4,072,262.

In [68]:
%%time
category_cols = ["Tested", "Sex", "Sanctioned", "Equipment", "BirthYearClass", 
                 "Event", "ParentFederation", "MeetCountry", "Federation", "State",
                 "MeetState", "AgeClass", "Country"]
dtypes = {col: "category" for col in category_cols}
dtypes["Place"] = "str"
dtypes["WeightClassKg"] = "str"
dtypes["Name"] = "str"

df = pd.read_csv(
    CSV_PATH, 
    dtype=dtypes,
    parse_dates=["Date"],
    )
df_shape = df.shape
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 4072262 entries, 0 to 4072261
Data columns (total 42 columns):
 #   Column            Dtype         
---  ------            -----         
 0   Name              str           
 1   Sex               category      
 2   Event             category      
 3   Equipment         category      
 4   Age               float64       
 5   AgeClass          str           
 6   BirthYearClass    category      
 7   Division          str           
 8   BodyweightKg      float64       
 9   WeightClassKg     str           
 10  Squat1Kg          float64       
 11  Squat2Kg          float64       
 12  Squat3Kg          float64       
 13  Squat4Kg          float64       
 14  Best3SquatKg      float64       
 15  Bench1Kg          float64       
 16  Bench2Kg          float64       
 17  Bench3Kg          float64       
 18  Bench4Kg          float64       
 19  Best3BenchKg      float64       
 20  Deadlift1Kg       float64       
 21  Deadlift2Kg       f